# STAGE 5.5 — FULL-FRAME CONTROLLED BASELINE

In [6]:
# ============================================================================
# STAGE 5.5 — FULL-FRAME CONTROLLED BASELINE
# ============================================================================
#
# Research question:
# Does the Hand-ROI representation improve recognition compared with
# the ORIGINAL FULL-FRAME representation?
#
# IMPORTANT:
# - Uses the frozen Stage 5.3 analysis population
# - ONE documented video is excluded because its prescribed sampled frame
#   cannot be decoded by the tested extraction methods
# - Uses the SAME 8 sampled frame indices defined by Stage 5.3
# - Uses the SAME MobileNetV3-Large architecture
# - Uses the SAME 960-D embedding representation
# - Uses the SAME prototype -> test cosine retrieval protocol
# - NO training
# - NO detector
# - NO ROI extraction
# - NO raw-video modification
#
# DOCUMENTED EXCLUSION:
# UID Xt6ujzVtikU_2
# Prescribed frame: 96
#
# Evidence:
# - Source video exists and is readable at container level
# - OpenCV sequential decoding reaches frame 95 but not frame 96
# - OpenCV direct seek to frame 96 fails
# - FFmpeg exact-frame extraction fails
# - FFprobe reports valid MPEG-4 metadata with 97 frames
#
# Therefore:
# Xt6ujzVtikU_2 is excluded from the controlled Stage 5.5 population.
#
# This exclusion is NOT treated as neighboring-frame substitution.
#
# ============================================================================

from pathlib import Path
import os
import json
import time
import hashlib
import warnings
import random

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn

from torchvision import models, transforms

warnings.filterwarnings("ignore")

print("=" * 100)
print("STAGE 5.5 — FULL-FRAME CONTROLLED BASELINE")
print("=" * 100)


# ============================================================================
# 5.5.1 — PATHS
# ============================================================================

PROJECT_ROOT = Path("/home/dipshikha/Music/cao")

RESEARCH_ROOT = PROJECT_ROOT / "CISLR_RESEARCH"

STAGE5_ROOT = (
    RESEARCH_ROOT
    / "audit"
    / "stage5_hand_roi"
)

# Frozen Stage 5.3 analysis-valid population
VALID_UID_MANIFEST = (
    STAGE5_ROOT
    / "analysis_valid_subset"
    / "stage5_3_analysis_valid_uid_manifest.csv"
)

# Stage 5.3 frame-level audit
FRAME_AUDIT = (
    STAGE5_ROOT
    / "roi_extraction"
    / "stage5_3_frame_level_extraction.csv"
)

# Original CISLR metadata
DATASET_CSV = PROJECT_ROOT / "dataset.csv"
PROTOTYPE_CSV = PROJECT_ROOT / "prototype.csv"
TEST_CSV = PROJECT_ROOT / "test.csv"

# Raw videos
VIDEO_ROOT = PROJECT_ROOT / "CISLR_v1.5-a_videos"

# Stage 5.5 output
OUTPUT_ROOT = (
    STAGE5_ROOT
    / "full_frame_baseline"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

EMBEDDING_DIR = OUTPUT_ROOT / "embeddings"
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

REPORT_DIR = OUTPUT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

PLOTS_DIR = OUTPUT_ROOT / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


print("\nPaths:")
print("Project root :", PROJECT_ROOT)
print("Valid UID    :", VALID_UID_MANIFEST)
print("Frame audit  :", FRAME_AUDIT)
print("Dataset      :", DATASET_CSV)
print("Prototype    :", PROTOTYPE_CSV)
print("Test         :", TEST_CSV)
print("Video root   :", VIDEO_ROOT)
print("Output       :", OUTPUT_ROOT)


# ============================================================================
# 5.5.2 — DOCUMENTED EXCLUSION
# ============================================================================

# One video has a prescribed Stage 5.3 frame that is not decodable.
#
# This is an explicit, frozen exclusion.
# No neighboring frame is substituted.

EXCLUDED_UIDS = {
    "Xt6ujzVtikU_2"
}

EXCLUSION_REASON = (
    "Prescribed Stage 5.3 frame 96 is not decodable by the tested "
    "OpenCV and FFmpeg extraction methods despite valid container-level "
    "video metadata."
)

EXCLUSION_FRAME = {
    "Xt6ujzVtikU_2": 96
}

print("\nDocumented Stage 5.5 exclusion:")
for uid in sorted(EXCLUDED_UIDS):
    print(
        f"  UID: {uid} | "
        f"prescribed frame: {EXCLUSION_FRAME.get(uid, 'unknown')}"
    )
    print(f"  Reason: {EXCLUSION_REASON}")


# ============================================================================
# 5.5.3 — VERIFY INPUTS
# ============================================================================

required_paths = [
    VALID_UID_MANIFEST,
    FRAME_AUDIT,
    DATASET_CSV,
    PROTOTYPE_CSV,
    TEST_CSV,
    VIDEO_ROOT,
]

missing = [
    str(p)
    for p in required_paths
    if not p.exists()
]

if missing:
    raise FileNotFoundError(
        "The following required inputs are missing:\n"
        + "\n".join(missing)
    )

print("\n✓ All required Stage 5.5 inputs found.")


# ============================================================================
# 5.5.4 — LOAD FROZEN ANALYSIS POPULATION
# ============================================================================

valid_manifest = pd.read_csv(
    VALID_UID_MANIFEST
)

frame_audit = pd.read_csv(
    FRAME_AUDIT
)

print("\nLoaded:")
print("Valid UID manifest :", valid_manifest.shape)
print("Stage 5.3 frame audit:", frame_audit.shape)

if "uid" not in valid_manifest.columns:
    raise ValueError(
        "Frozen UID manifest must contain 'uid'."
    )

if "uid" not in frame_audit.columns:
    raise ValueError(
        "Stage 5.3 frame audit must contain 'uid'."
    )


# ----------------------------------------------------------------------------
# Frozen UID set BEFORE exclusion
# ----------------------------------------------------------------------------

frozen_uids = (
    valid_manifest["uid"]
    .astype(str)
    .drop_duplicates()
    .tolist()
)

frozen_uid_set = set(frozen_uids)

print("\nOriginal frozen Stage 5.3 population:")
print("Videos:", len(frozen_uids))

if len(frozen_uids) != 6628:
    raise ValueError(
        f"Expected exactly 6628 frozen analysis-valid videos, "
        f"but found {len(frozen_uids)}."
    )


# ----------------------------------------------------------------------------
# Verify documented exclusion exists in frozen population
# ----------------------------------------------------------------------------

missing_exclusions = (
    EXCLUDED_UIDS - frozen_uid_set
)

if missing_exclusions:
    raise ValueError(
        "The documented exclusion UID(s) are not present in the "
        f"frozen Stage 5.3 population: {missing_exclusions}"
    )

print(
    f"✓ Documented exclusion UID(s) confirmed in frozen population: "
    f"{len(EXCLUDED_UIDS)}"
)


# ----------------------------------------------------------------------------
# Apply exclusion
# ----------------------------------------------------------------------------

valid_uids = [
    uid
    for uid in frozen_uids
    if uid not in EXCLUDED_UIDS
]

valid_uid_set = set(valid_uids)

print("\nStage 5.5 controlled population:")
print("Frozen Stage 5.3 videos :", len(frozen_uids))
print("Documented exclusions   :", len(EXCLUDED_UIDS))
print("Stage 5.5 videos        :", len(valid_uids))

EXPECTED_STAGE5_5_VIDEOS = 6627

if len(valid_uids) != EXPECTED_STAGE5_5_VIDEOS:
    raise ValueError(
        f"Expected {EXPECTED_STAGE5_5_VIDEOS} Stage 5.5 videos "
        f"after documented exclusion, but found {len(valid_uids)}."
    )

if EXCLUDED_UIDS.intersection(valid_uid_set):
    raise RuntimeError(
        "Excluded UID(s) are still present in the Stage 5.5 population."
    )

print("✓ Stage 5.5 population frozen at 6,627 videos.")


# ============================================================================
# 5.5.5 — SAVE EXCLUSION AUDIT
# ============================================================================

exclusion_rows = []

for uid in sorted(EXCLUDED_UIDS):

    exclusion_rows.append({
        "uid": uid,
        "excluded_from_stage": "5.5",
        "prescribed_frame_index": EXCLUSION_FRAME.get(uid),
        "reason": EXCLUSION_REASON,
        "neighbor_frame_substitution": False,
        "raw_video_modified": False,
    })

exclusion_df = pd.DataFrame(
    exclusion_rows
)

exclusion_df.to_csv(
    REPORT_DIR
    / "stage5_5_documented_exclusions.csv",
    index=False
)

print(
    "\n✓ Stage 5.5 exclusion audit saved:"
)

print(
    REPORT_DIR
    / "stage5_5_documented_exclusions.csv"
)


# ============================================================================
# 5.5.6 — RECOVER EXACT 8 FRAME INDICES FROM STAGE 5.3
# ============================================================================

required_frame_columns = [
    "uid",
    "sample_number",
    "frame_index",
    "video_path",
]

missing_columns = [
    c
    for c in required_frame_columns
    if c not in frame_audit.columns
]

if missing_columns:
    raise ValueError(
        f"Stage 5.3 frame audit missing columns: {missing_columns}"
    )


# ----------------------------------------------------------------------------
# Match ONLY the Stage 5.5 controlled population
# ----------------------------------------------------------------------------

matched_frames = frame_audit[
    frame_audit["uid"]
    .astype(str)
    .isin(valid_uid_set)
].copy()

matched_frames["uid"] = (
    matched_frames["uid"]
    .astype(str)
)

matched_frames = (
    matched_frames
    .sort_values(
        ["uid", "sample_number"]
    )
    .reset_index(drop=True)
)

print(
    "\nMatched Stage 5.3 frame records:",
    len(matched_frames)
)

EXPECTED_FRAMES = len(valid_uids) * 8

if len(matched_frames) != EXPECTED_FRAMES:
    raise ValueError(
        f"Expected {EXPECTED_FRAMES} matched frame records "
        f"for {len(valid_uids)} videos × 8 frames, "
        f"but found {len(matched_frames)}."
    )

frames_per_video = (
    matched_frames
    .groupby("uid")
    .size()
)

bad_counts = frames_per_video[
    frames_per_video != 8
]

if len(bad_counts) > 0:
    print("\nVideos with incorrect frame counts:")
    print(bad_counts.head(20))

    raise ValueError(
        f"{len(bad_counts)} videos do not contain exactly "
        "8 sampled frames."
    )

print(
    f"✓ Confirmed {len(valid_uids)} videos × 8 frames "
    f"= {EXPECTED_FRAMES} frame records."
)


# ============================================================================
# 5.5.7 — VERIFY FRAME SAMPLING
# ============================================================================

print("\nStage 5.3 sampling structure:")

sample_distribution = (
    matched_frames["sample_number"]
    .value_counts()
    .sort_index()
)

print(sample_distribution)

sample_values = list(
    sample_distribution.index
)

if sample_values not in (
    list(range(0, 8)),
    list(range(1, 9)),
):
    raise ValueError(
        "Expected sample_number values 0–7 or 1–8."
    )

if not all(
    sample_distribution.values == len(valid_uids)
):
    raise ValueError(
        "Every sample number must occur exactly once per video."
    )

print(
    "✓ Exact Stage 5.3 8-frame sampling recovered."
)


# ============================================================================
# 5.5.8 — VERIFY EXCLUDED VIDEO IS ABSENT
# ============================================================================

excluded_still_present = (
    set(
        matched_frames["uid"]
        .astype(str)
    )
    .intersection(EXCLUDED_UIDS)
)

if excluded_still_present:
    raise RuntimeError(
        "Excluded video(s) still appear in Stage 5.5 frame records: "
        f"{excluded_still_present}"
    )

print(
    "✓ Xt6ujzVtikU_2 is completely excluded from Stage 5.5."
)


# ============================================================================
# 5.5.9 — VERIFY RAW VIDEO PATHS
# ============================================================================

def resolve_video_path(row):

    """
    Resolve the original CISLR video path.

    Stage 5.3 already stores the original video_path,
    so we first use that.

    If it is absolute and exists, use it directly.

    Otherwise, resolve by basename under VIDEO_ROOT.

    Finally, try UID-based filenames.
    """

    p = Path(
        str(row["video_path"])
    )

    if p.exists():
        return p

    candidate = (
        VIDEO_ROOT
        / p.name
    )

    if candidate.exists():
        return candidate

    uid = str(row["uid"])

    candidates = list(
        VIDEO_ROOT.glob(
            f"{uid}*"
        )
    )

    if len(candidates) == 1:
        return candidates[0]

    return None


video_lookup = (
    matched_frames[
        ["uid", "video_path"]
    ]
    .drop_duplicates("uid")
    .copy()
)

video_lookup["resolved_video_path"] = (
    video_lookup.apply(
        resolve_video_path,
        axis=1
    )
)

missing_videos = video_lookup[
    video_lookup["resolved_video_path"].isna()
]

print("\nOriginal videos resolved:")
print(
    f"{len(video_lookup) - len(missing_videos)}/"
    f"{len(video_lookup)}"
)

if len(missing_videos) > 0:

    print("\nFirst missing examples:")
    print(
        missing_videos.head(10)
    )

    raise FileNotFoundError(
        f"{len(missing_videos)} Stage 5.5 videos "
        "could not be resolved."
    )

print(
    f"✓ All {len(valid_uids)} Stage 5.5 videos resolved."
)

video_path_map = dict(
    zip(
        video_lookup["uid"],
        video_lookup[
            "resolved_video_path"
        ]
    )
)


# ============================================================================
# 5.5.10 — LOAD OFFICIAL PROTOTYPE / TEST METADATA
# ============================================================================

prototype = pd.read_csv(
    PROTOTYPE_CSV
)

test = pd.read_csv(
    TEST_CSV
)

prototype["uid"] = (
    prototype["uid"]
    .astype(str)
)

test["uid"] = (
    test["uid"]
    .astype(str)
)

print("\nOfficial retrieval metadata:")
print(
    "Prototype rows:",
    len(prototype)
)

print(
    "Test rows     :",
    len(test)
)


# ----------------------------------------------------------------------------
# Keep ONLY Stage 5.5 controlled population
# ----------------------------------------------------------------------------

prototype_stage5 = prototype[
    prototype["uid"]
    .isin(valid_uid_set)
].copy()

test_stage5 = test[
    test["uid"]
    .isin(valid_uid_set)
].copy()

print("\nStage 5.5 matched retrieval population:")
print(
    "Prototype videos:",
    len(prototype_stage5)
)

print(
    "Test videos     :",
    len(test_stage5)
)

if len(prototype_stage5) == 0:
    raise ValueError(
        "No prototype videos remain in Stage 5.5 population."
    )

if len(test_stage5) == 0:
    raise ValueError(
        "No test videos remain in Stage 5.5 population."
    )


# ============================================================================
# 5.5.11 — LOAD MOBILENETV3-LARGE
# ============================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", DEVICE)


# ----------------------------------------------------------------------------
# Load architecture
# ----------------------------------------------------------------------------

weights = (
    models.MobileNet_V3_Large_Weights.DEFAULT
)

mobilenet = models.mobilenet_v3_large(
    weights=weights
)


# ----------------------------------------------------------------------------
# MobileNetV3-Large embedding model
# ----------------------------------------------------------------------------

class MobileNetV3LargeEmbedding(
    nn.Module
):

    def __init__(
        self,
        backbone
    ):

        super().__init__()

        self.features = (
            backbone.features
        )

        self.avgpool = (
            backbone.avgpool
        )

    def forward(
        self,
        x
    ):

        x = self.features(x)

        x = self.avgpool(x)

        x = torch.flatten(
            x,
            1
        )

        return x


embedding_model = (
    MobileNetV3LargeEmbedding(
        mobilenet
    )
)

embedding_model = (
    embedding_model
    .to(DEVICE)
)

embedding_model.eval()


# ----------------------------------------------------------------------------
# Verify embedding dimension
# ----------------------------------------------------------------------------

with torch.no_grad():

    dummy = torch.zeros(
        1,
        3,
        224,
        224,
        device=DEVICE
    )

    dummy_embedding = (
        embedding_model(dummy)
    )

embedding_dim = (
    dummy_embedding.shape[1]
)

print(
    "\nEmbedding dimension:",
    embedding_dim
)

if embedding_dim != 960:
    raise ValueError(
        f"Expected 960-D MobileNetV3-Large embedding, "
        f"got {embedding_dim}."
    )

print(
    "✓ Confirmed 960-D embedding."
)


# ============================================================================
# 5.5.12 — IMAGE PREPROCESSING
# ============================================================================

preprocess = transforms.Compose([

    transforms.ToPILImage(),

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

print(
    "\n✓ MobileNetV3-Large ImageNet preprocessing configured."
)


# ============================================================================
# 5.5.13 — FRAME EXTRACTION FUNCTION
# ============================================================================

def read_exact_frame(
    video_path,
    frame_index
):

    """
    Read the exact requested frame.

    IMPORTANT:
    - No neighboring-frame substitution.
    - No hand detector.
    - No ROI crop.
    - Original full frame is returned.
    """

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        return None

    try:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(frame_index)
        )

        ret, frame = cap.read()

        if (
            not ret
            or frame is None
        ):
            return None

        return frame

    finally:

        cap.release()


# ============================================================================
# 5.5.14 — VIDEO EMBEDDING FUNCTION
# ============================================================================

@torch.no_grad()
def extract_video_embedding(
    uid,
    rows
):

    frames = []

    for _, row in rows.iterrows():

        frame_index = int(
            row["frame_index"]
        )

        video_path = (
            video_path_map[uid]
        )

        frame = read_exact_frame(
            video_path,
            frame_index
        )

        if frame is None:

            raise RuntimeError(
                f"Could not read exact frame "
                f"{frame_index} for UID {uid}"
            )

        # OpenCV BGR → RGB
        frame_rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        tensor = preprocess(
            frame_rgb
        )

        frames.append(
            tensor
        )

    if len(frames) != 8:

        raise RuntimeError(
            f"{uid}: expected 8 frames, "
            f"got {len(frames)}"
        )

    batch = torch.stack(
        frames,
        dim=0
    ).to(DEVICE)

    frame_embeddings = (
        embedding_model(batch)
    )

    # ------------------------------------------------------------------------
    # 8 frame embeddings → one video embedding
    # ------------------------------------------------------------------------
    #
    # Mean pooling is performed before final L2 normalization.
    #
    # ------------------------------------------------------------------------

    video_embedding = (
        frame_embeddings.mean(
            dim=0
        )
    )

    video_embedding = (
        torch.nn.functional.normalize(
            video_embedding,
            p=2,
            dim=0
        )
    )

    return (
        video_embedding
        .cpu()
        .numpy()
    )


# ============================================================================
# 5.5.15 — EXTRACT FULL-FRAME EMBEDDINGS
# ============================================================================

print("\n" + "=" * 100)
print("EXTRACTING FULL-FRAME VIDEO EMBEDDINGS")
print("=" * 100)

print(
    "\nStage 5.5 videos:",
    len(valid_uids)
)

print(
    "Documented exclusions:",
    len(EXCLUDED_UIDS)
)

print(
    "Frames:",
    EXPECTED_FRAMES
)

print(
    "Embedding dimension:",
    embedding_dim
)

start_time = time.time()

video_embeddings = {}

embedding_errors = []

grouped = matched_frames.groupby(
    "uid",
    sort=False
)

total_videos = len(grouped)

for video_number, (
    uid,
    rows
) in enumerate(
    grouped,
    start=1
):

    try:

        emb = extract_video_embedding(
            uid,
            rows
        )

        video_embeddings[uid] = emb

    except Exception as e:

        embedding_errors.append({

            "uid":
                uid,

            "error":
                str(e)

        })

    if (
        video_number == 1
        or video_number % 100 == 0
        or video_number == total_videos
    ):

        elapsed = (
            time.time()
            - start_time
        )

        rate = (
            video_number
            / max(
                elapsed,
                1e-9
            )
        )

        print(
            f"[{video_number:4d}/{total_videos}] "
            f"{rate:.2f} videos/s | "
            f"successful={len(video_embeddings)} | "
            f"errors={len(embedding_errors)}"
        )


elapsed = (
    time.time()
    - start_time
)

print("\nExtraction complete.")

print(
    f"Successful videos: "
    f"{len(video_embeddings)}"
)

print(
    f"Failed videos:     "
    f"{len(embedding_errors)}"
)

print(
    f"Elapsed time:      "
    f"{elapsed / 60:.2f} min"
)


# ============================================================================
# 5.5.16 — EMBEDDING ERROR POLICY
# ============================================================================

if embedding_errors:

    error_df = pd.DataFrame(
        embedding_errors
    )

    error_df.to_csv(
        REPORT_DIR
        / "stage5_5_embedding_errors.csv",
        index=False
    )

    print(
        "\nERROR: Unexpected embedding failures detected."
    )

    print(
        error_df.to_string(
            index=False
        )
    )

    raise RuntimeError(
        f"{len(embedding_errors)} unexpected videos failed "
        "full-frame embedding extraction. "
        "Successful embeddings were kept in memory and "
        "the error audit was saved."
    )


# ----------------------------------------------------------------------------
# Verify exact success count
# ----------------------------------------------------------------------------

if len(video_embeddings) != len(valid_uids):

    raise RuntimeError(
        f"Expected {len(valid_uids)} successful embeddings, "
        f"but obtained {len(video_embeddings)}."
    )

print(
    f"✓ All {len(valid_uids)} Stage 5.5 videos successfully embedded."
)


# ============================================================================
# 5.5.17 — VERIFY EMBEDDING MATRIX
# ============================================================================

embedding_matrix = np.stack([

    video_embeddings[uid]

    for uid in valid_uids

])

print("\nEmbedding matrix:")

print(
    "Shape:",
    embedding_matrix.shape
)

expected_embedding_shape = (
    len(valid_uids),
    960
)

if (
    embedding_matrix.shape
    != expected_embedding_shape
):

    raise ValueError(
        f"Unexpected embedding matrix shape: "
        f"{embedding_matrix.shape}"
    )


# ----------------------------------------------------------------------------
# Verify L2 normalization
# ----------------------------------------------------------------------------

embedding_norms = np.linalg.norm(
    embedding_matrix,
    axis=1
)

print(
    "Mean L2 norm:",
    embedding_norms.mean()
)

print(
    "Minimum L2 norm:",
    embedding_norms.min()
)

print(
    "Maximum L2 norm:",
    embedding_norms.max()
)

if not np.allclose(
    embedding_norms,
    1.0,
    atol=1e-4
):

    raise ValueError(
        "Embeddings are not properly L2-normalized."
    )

print(
    "✓ All video embeddings are L2-normalized."
)


# ============================================================================
# 5.5.18 — SAVE FULL-FRAME EMBEDDINGS
# ============================================================================

embedding_matrix_path = (
    EMBEDDING_DIR
    / "stage5_5_full_frame_embeddings.npy"
)

np.save(
    embedding_matrix_path,
    embedding_matrix
)

embedding_uid_manifest = pd.DataFrame({

    "uid":
        valid_uids,

    "embedding_index":
        np.arange(
            len(valid_uids)
        )

})

embedding_uid_manifest_path = (
    EMBEDDING_DIR
    / "stage5_5_full_frame_embedding_uid_manifest.csv"
)

embedding_uid_manifest.to_csv(
    embedding_uid_manifest_path,
    index=False
)

print(
    "\n✓ Full-frame embedding matrix saved:"
)

print(
    embedding_matrix_path
)

print(
    "✓ Embedding UID manifest saved:"
)

print(
    embedding_uid_manifest_path
)


# ============================================================================
# 5.5.19 — CREATE UID → EMBEDDING LOOKUP
# ============================================================================

embedding_lookup = {

    uid:
        video_embeddings[uid]

    for uid in valid_uids

}


# ============================================================================
# 5.5.20 — PREPARE OFFICIAL PROTOTYPE / TEST RETRIEVAL
# ============================================================================

prototype_stage5 = prototype_stage5[
    prototype_stage5["uid"]
    .isin(embedding_lookup)
].copy()

test_stage5 = test_stage5[
    test_stage5["uid"]
    .isin(embedding_lookup)
].copy()

print(
    "\nRetrieval population:"
)

print(
    "Prototype:",
    len(prototype_stage5)
)

print(
    "Test     :",
    len(test_stage5)
)


# ============================================================================
# 5.5.21 — BUILD PROTOTYPE EMBEDDINGS
# ============================================================================

prototype_uids = (
    prototype_stage5["uid"]
    .tolist()
)

test_uids = (
    test_stage5["uid"]
    .tolist()
)

prototype_embeddings = np.stack([

    embedding_lookup[uid]

    for uid in prototype_uids

])

test_embeddings = np.stack([

    embedding_lookup[uid]

    for uid in test_uids

])

print(
    "\nPrototype embedding matrix:"
)

print(
    prototype_embeddings.shape
)

print(
    "\nTest embedding matrix:"
)

print(
    test_embeddings.shape
)


# ============================================================================
# 5.5.22 — COSINE SIMILARITY
# ============================================================================

#
# Because all embeddings are L2-normalized:
#
# cosine_similarity(a,b) = a · b
#
# Therefore matrix multiplication gives cosine similarity.
#

similarity_matrix = (
    test_embeddings
    @ prototype_embeddings.T
)

print(
    "\nSimilarity matrix:",
    similarity_matrix.shape
)


# ============================================================================
# 5.5.23 — RANKING
# ============================================================================

ranking = np.argsort(
    -similarity_matrix,
    axis=1
)

sorted_scores = np.take_along_axis(
    similarity_matrix,
    ranking,
    axis=1
)


# ============================================================================
# 5.5.24 — MAP PROTOTYPE UID → GLOSS
# ============================================================================

prototype_uid_to_gloss = dict(

    zip(
        prototype_stage5["uid"],
        prototype_stage5["gloss"].astype(str)
    )

)

test_uid_to_gloss = dict(

    zip(
        test_stage5["uid"],
        test_stage5["gloss"].astype(str)
    )

)


# ============================================================================
# 5.5.25 — OFFICIAL RETRIEVAL METRICS
# ============================================================================

top1_hits = []
top5_hits = []
top10_hits = []

ranks = []

retrieval_rows = []


for i, test_uid in enumerate(
    test_uids
):

    true_gloss = (
        test_uid_to_gloss[
            test_uid
        ]
    )

    ranked_indices = (
        ranking[i]
    )

    ranked_uids = [

        prototype_uids[j]

        for j in ranked_indices

    ]

    ranked_glosses = [

        prototype_uid_to_gloss[uid]

        for uid in ranked_uids

    ]


    # ------------------------------------------------------------------------
    # Find first prototype whose gloss matches the test gloss.
    # ------------------------------------------------------------------------

    matching_positions = [

        rank_position

        for rank_position, gloss
        in enumerate(ranked_glosses)

        if gloss == true_gloss

    ]


    if matching_positions:

        first_match = (
            matching_positions[0]
            + 1
        )

    else:

        first_match = (
            len(ranked_glosses)
            + 1
        )


    ranks.append(
        first_match
    )

    top1_hits.append(
        first_match <= 1
    )

    top5_hits.append(
        first_match <= 5
    )

    top10_hits.append(
        first_match <= 10
    )


    retrieval_rows.append({

        "test_uid":
            test_uid,

        "true_gloss":
            true_gloss,

        "rank":
            first_match,

        "top1_correct":
            first_match <= 1,

        "top5_correct":
            first_match <= 5,

        "top10_correct":
            first_match <= 10,

        "top1_predicted_uid":
            ranked_uids[0],

        "top1_predicted_gloss":
            ranked_glosses[0],

        "top1_similarity":
            float(
                sorted_scores[i, 0]
            ),

        "top5_uids":
            "|".join(
                ranked_uids[:5]
            ),

        "top5_glosses":
            "|".join(
                ranked_glosses[:5]
            ),

        "top10_uids":
            "|".join(
                ranked_uids[:10]
            ),

        "top10_glosses":
            "|".join(
                ranked_glosses[:10]
            ),

    })


# ============================================================================
# 5.5.26 — CALCULATE FINAL METRICS
# ============================================================================

ranks = np.asarray(
    ranks,
    dtype=np.int64
)

top1 = np.mean(
    np.asarray(
        top1_hits
    )
)

top5 = np.mean(
    np.asarray(
        top5_hits
    )
)

recall10 = np.mean(
    np.asarray(
        top10_hits
    )
)

mrr = np.mean(
    1.0 / ranks
)

mean_rank = np.mean(
    ranks
)

median_rank = np.median(
    ranks
)


# ============================================================================
# 5.5.27 — PRINT RESULTS
# ============================================================================

print("\n")

print("=" * 100)

print(
    "STAGE 5.5 — FULL-FRAME CONTROLLED BASELINE RESULTS"
)

print("=" * 100)

print(
    f"\nOriginal frozen Stage 5.3 videos : "
    f"{len(frozen_uids)}"
)

print(
    f"Documented exclusions            : "
    f"{len(EXCLUDED_UIDS)}"
)

print(
    f"Stage 5.5 analysis videos        : "
    f"{len(valid_uids)}"
)

print(
    f"Prototype videos                 : "
    f"{len(prototype_uids)}"
)

print(
    f"Test videos                      : "
    f"{len(test_uids)}"
)

print(
    f"Embedding dimension              : "
    f"{embedding_dim}"
)

print(
    "Frames per video                 : 8"
)

print(
    "Backbone                         : "
    "MobileNetV3-Large"
)

print(
    "Representation                   : "
    "Full Frame"
)

print(
    "Retrieval                        : "
    "Prototype → Test"
)

print(
    "Similarity                       : "
    "Cosine"
)

print("\n--- RETRIEVAL METRICS ---")

print(
    f"Top-1 Accuracy       : "
    f"{top1:.4%}"
)

print(
    f"Top-5 Accuracy       : "
    f"{top5:.4%}"
)

print(
    f"Recall@10            : "
    f"{recall10:.4%}"
)

print(
    f"MRR                  : "
    f"{mrr:.4f}"
)

print(
    f"Mean Rank            : "
    f"{mean_rank:.2f}"
)

print(
    f"Median Rank          : "
    f"{median_rank:.2f}"
)


# ============================================================================
# 5.5.28 — SAVE RETRIEVAL RESULTS
# ============================================================================

retrieval_df = pd.DataFrame(
    retrieval_rows
)

retrieval_results_path = (
    REPORT_DIR
    / "stage5_5_full_frame_retrieval_results.csv"
)

retrieval_df.to_csv(
    retrieval_results_path,
    index=False
)

print(
    "\n✓ Per-test retrieval results saved:"
)

print(
    retrieval_results_path
)


# ============================================================================
# 5.5.29 — SAVE SUMMARY JSON
# ============================================================================

summary = {

    "stage":
        "5.5",

    "name":
        "Full-Frame Controlled Baseline",

    "research_question":
        "Does hand-ROI input improve recognition compared with full-frame input?",

    "analysis_population":
        "Frozen Stage 5.3 analysis-valid subset minus one documented "
        "frame-undecodable video",

    "original_frozen_analysis_videos":
        len(frozen_uids),

    "documented_exclusions":
        len(EXCLUDED_UIDS),

    "analysis_videos":
        len(valid_uids),

    "excluded_uids":
        sorted(
            EXCLUDED_UIDS
        ),

    "exclusion_reason":
        EXCLUSION_REASON,

    "excluded_prescribed_frames":
        EXCLUSION_FRAME,

    "frames_per_video":
        8,

    "total_frames":
        EXPECTED_FRAMES,

    "prototype_videos":
        len(prototype_uids),

    "test_videos":
        len(test_uids),

    "backbone":
        "MobileNetV3-Large",

    "embedding_dimension":
        int(embedding_dim),

    "representation":
        "Full Frame",

    "frame_sampling":
        "Exact Stage 5.3 frame indices",

    "pooling":
        "Mean pooling across 8 frame embeddings",

    "normalization":
        "L2",

    "retrieval":
        "Official prototype-to-test retrieval",

    "similarity":
        "Cosine",

    "top1":
        float(top1),

    "top5":
        float(top5),

    "recall_at_10":
        float(recall10),

    "mrr":
        float(mrr),

    "mean_rank":
        float(mean_rank),

    "median_rank":
        float(median_rank),

    "extraction_time_minutes":
        float(
            elapsed / 60.0
        ),

    "raw_videos_modified":
        False,

    "stage4_modified":
        False,

    "stage5_1_modified":
        False,

    "stage5_2_modified":
        False,

    "stage5_3_modified":
        False,

    "stage5_4_modified":
        False,

    "training_performed":
        False,

    "detector_used":
        None,

    "roi_extraction_performed":
        False,

    "neighbor_frame_substitution":
        False,

}

summary_path = (
    REPORT_DIR
    / "stage5_5_full_frame_baseline_summary.json"
)

with open(
    summary_path,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

print(
    "\n✓ Stage 5.5 summary JSON saved:"
)

print(
    summary_path
)


# ============================================================================
# 5.5.30 — SAVE PROTOCOL
# ============================================================================

protocol = {

    "stage":
        "5.5",

    "protocol_name":
        "Full-Frame Controlled Baseline",

    "population":
        "Frozen Stage 5.3 analysis-valid UID manifest "
        "minus documented frame-undecodable video",

    "original_number_of_videos":
        len(frozen_uids),

    "number_of_documented_exclusions":
        len(EXCLUDED_UIDS),

    "number_of_videos":
        len(valid_uids),

    "excluded_uids":
        sorted(
            EXCLUDED_UIDS
        ),

    "exclusion_reason":
        EXCLUSION_REASON,

    "excluded_prescribed_frames":
        EXCLUSION_FRAME,

    "frames_per_video":
        8,

    "total_frame_records":
        EXPECTED_FRAMES,

    "frame_source":
        "Original CISLR videos",

    "frame_indices":
        "Exactly those recorded in Stage 5.3 frame-level audit",

    "input_representation":
        "Original full video frame",

    "resize":
        "224x224",

    "color":
        "RGB",

    "normalization":
        "ImageNet mean/std",

    "backbone":
        "MobileNetV3-Large",

    "embedding_dimension":
        960,

    "frame_embedding_pooling":
        "Mean",

    "video_embedding_normalization":
        "L2",

    "retrieval_protocol":
        "Official prototype-to-test retrieval",

    "similarity_metric":
        "Cosine similarity",

    "training":
        False,

    "hand_detector":
        False,

    "roi_extraction":
        False,

    "neighbor_frame_substitution":
        False,

    "raw_video_modification":
        False,

}

protocol_path = (
    REPORT_DIR
    / "stage5_5_full_frame_protocol.json"
)

with open(
    protocol_path,
    "w"
) as f:

    json.dump(
        protocol,
        f,
        indent=2
    )

print(
    "\n✓ Stage 5.5 protocol saved:"
)

print(
    protocol_path
)


# ============================================================================
# 5.5.31 — FINAL AUDIT
# ============================================================================

print("\n")

print("=" * 100)

print(
    "STAGE 5.5 FINAL AUDIT"
)

print("=" * 100)

print(
    f"✓ Original Stage 5.3 videos     : "
    f"{len(frozen_uids)}"
)

print(
    f"✓ Documented exclusions         : "
    f"{len(EXCLUDED_UIDS)}"
)

print(
    f"✓ Controlled videos used        : "
    f"{len(valid_uids)}"
)

print(
    f"✓ Frames per video              : 8"
)

print(
    f"✓ Total frame records           : "
    f"{EXPECTED_FRAMES}"
)

print(
    "✓ Full-frame representation     : YES"
)

print(
    "✓ MobileNetV3-Large             : YES"
)

print(
    "✓ 960-D embeddings              : YES"
)

print(
    "✓ Mean frame pooling            : YES"
)

print(
    "✓ L2 normalization              : YES"
)

print(
    "✓ Prototype → Test              : YES"
)

print(
    "✓ Cosine similarity             : YES"
)

print(
    "✓ Training performed            : NO"
)

print(
    "✓ Hand detector used            : NO"
)

print(
    "✓ ROI extraction performed      : NO"
)

print(
    "✓ Neighbor substitution         : NO"
)

print(
    "✓ Raw videos modified           : NO"
)

print(
    "✓ Stage 4 modified              : NO"
)

print(
    "✓ Stage 5.1 modified            : NO"
)

print(
    "✓ Stage 5.2 modified            : NO"
)

print(
    "✓ Stage 5.3 modified            : NO"
)

print(
    "✓ Stage 5.4 modified            : NO"
)

print(
    "\nExcluded UID:"
)

for uid in sorted(
    EXCLUDED_UIDS
):

    print(
        f"  - {uid} "
        f"(prescribed frame "
        f"{EXCLUSION_FRAME.get(uid)})"
    )

print(
    "\nArtifacts:"
)

print(
    OUTPUT_ROOT
)

print(
    "\nSTAGE 5.5 COMPLETE — "
    "FULL-FRAME BASELINE FROZEN"
)

print("=" * 100)

STAGE 5.5 — FULL-FRAME CONTROLLED BASELINE

Paths:
Project root : /home/dipshikha/Music/cao
Valid UID    : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/analysis_valid_subset/stage5_3_analysis_valid_uid_manifest.csv
Frame audit  : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/roi_extraction/stage5_3_frame_level_extraction.csv
Dataset      : /home/dipshikha/Music/cao/dataset.csv
Prototype    : /home/dipshikha/Music/cao/prototype.csv
Test         : /home/dipshikha/Music/cao/test.csv
Video root   : /home/dipshikha/Music/cao/CISLR_v1.5-a_videos
Output       : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage5_hand_roi/full_frame_baseline

Documented Stage 5.5 exclusion:
  UID: Xt6ujzVtikU_2 | prescribed frame: 96
  Reason: Prescribed Stage 5.3 frame 96 is not decodable by the tested OpenCV and FFmpeg extraction methods despite valid container-level video metadata.

✓ All required Stage 5.5 inputs found.

Loaded:
Valid UID manifest : (6628, 4)
Stage 5.3 